# 00 — Project contract

This notebook freezes the product claim before any performance result is seen.
The product is **Telecom-only and PON-first**. “Company-agnostic” means that
operators map their native fields into vendor-neutral Telecom semantics and
calibrate locally; it does not mean pooling incompatible PON, RAN and backbone
measurements into one fitted model.

Run this notebook first after a kernel restart. It reads version-controlled
configuration only and does not open telemetry or evaluation truth.


## 1. Load the versioned contract


In [ ]:
from pathlib import Path
import os
import sys


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import display

from telco_anomaly.io import load_config

project = load_config("project", project_root=PROJECT_ROOT)
datasets = load_config("dataset", project_root=PROJECT_ROOT)
metrics = load_config("metric_registry", project_root=PROJECT_ROOT)
topology = load_config("topology", project_root=PROJECT_ROOT)
alert_policy = load_config("alert_policy", project_root=PROJECT_ROOT)

display(pd.Series({
    "project": project["project"],
    "primary_domain": project["primary_domain"],
    "primary_dataset": datasets["primary_dataset"],
    "configuration_version": project["version"],
}, name="value").to_frame())


## 2. Product boundary

The current product produces anomaly evidence, consolidates alerts into
incidents, and localises likely infrastructure scope. Fault classification,
failure prediction and causal root-cause analysis are deliberately outside the
present claim.


In [ ]:
scope = pd.DataFrame({
    "in_scope": project["objectives"],
})
excluded = pd.DataFrame({
    "not_in_scope": project["out_of_scope"],
})
display(scope)
display(excluded)

assert project["primary_domain"] == "fixed_access_pon"
assert "failure_prediction" in project["out_of_scope"]
assert "topology_localisation" in project["objectives"]


## 3. Company independence and evidence levels


In [ ]:
display(pd.DataFrame({
    "company_agnostic_requirement": project["company_agnostic_means"]
}))

evidence = pd.DataFrame([
    {"level": name, "claim_supported": claim}
    for name, claim in project["claim_levels"].items()
])
display(evidence)

dataset_roles = pd.DataFrame([
    {
        "dataset": name,
        "role": definition["role"],
        "labelled": definition["labelled"],
        "evidence_level": definition["evidence_level"],
        "pooled_with_other_sources": False,
    }
    for name, definition in datasets["datasets"].items()
])
display(dataset_roles)


## 4. Model-safe vocabulary

The metric registry is authored, not learned. It defines how each Telecom
measurement is interpreted without looking at fault labels. Public datasets
receive their own adapters and metric packs; their raw columns are never
silently treated as equivalent to PON fields.


In [ ]:
metric_registry = pd.DataFrame(metrics["metrics"])
display(metric_registry[[
    "native_field", "metric_id", "measurement_kind", "unit", "direction",
    "transform", "expected_cadence_seconds", "seasonality_candidate",
]])

assert metric_registry["metric_id"].is_unique
assert not metric_registry["metric_id"].str.contains(
    r"fault|ticket|label|truth", case=False, regex=True
).any()
display(pd.DataFrame([
    {"native_field": field, **definition}
    for field, definition in topology["memberships"].items()
]))


## 5. Evaluation firewall and acceptance rules


In [ ]:
display(pd.DataFrame({
    "prohibited_model_field": project["prohibited_model_fields"]
}))
display(pd.Series({
    "threshold_method": alert_policy["thresholds"]["method"],
    "false_incident_budget_per_entity_day": (
        alert_policy["workload"]["false_incidents_per_entity_day"]
    ),
    "holdout_open_by_default": alert_policy["holdout_default_open"],
    "fail_closed": alert_policy["selection"]["fail_closed_when_no_candidate_meets_budget"],
}, name="policy").to_frame())

assert datasets["rules"]["pool_raw_datasets"] is False
assert datasets["rules"]["fit_one_model_across_metric_packs"] is False
assert datasets["rules"]["holdout_may_influence_model_choice"] is False
assert alert_policy["holdout_default_open"] is False
print("PASS — the project boundary and evidence policy are internally consistent")


## 6. Research sequence

`01` audits observable source data. `02` translates it to the canonical
contract. `03` freezes chronological splits and physically separates truth.
`04` uses calibration telemetry only. Feature engineering and models begin in
Notebook `05`; locked evaluation does not happen until Notebook `10`.
